# AI Operations Manager Agent

This notebook implements an agentic AI system that simulates an Operations Manager. It orchestrates multiple specialized AI agents to autonomously handle a business task, demonstrating modern agentic patterns like parallel execution, agent-as-tool, and handoffs.

## 1. Environment Setup

Load necessary libraries and configure the OpenAI API key.

In [ ]:
import osimport asyncioimport jsonfrom openai import OpenAIfrom openai.tools.function_tools import function_toolfrom dotenv import load_dotenv
# Load environment variablesload_dotenv()
# Initialize OpenAI clientclient = OpenAI()

## 2. Agent Definitions

Define the specialized agents. Each agent is an asynchronous function with a specific role and system prompt.

In [ ]:
async def finance_agent(task: str):    """Analyzes the financial implications of a task."""    system_prompt = 'You are a finance expert. Analyze the following task from a financial perspective, focusing on cost, budget, and ROI.'    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": task}]
    )    return {"agent": "Finance", "analysis": response.choices[0].message.content}
async def hr_agent(task: str):    """Analyzes the human resources implications of a task."""    system_prompt = 'You are an HR expert. Analyze the task from a staffing, and internal communications perspective.'    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": task}]
    )    return {"agent": "HR", "analysis": response.choices[0].message.content}
async def engineering_agent(task: str):    """Analyzes the technical implementation of a task."""    system_prompt = 'You are an engineering lead. Provide a high-level technical execution plan for the task.'    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": task}]
    )    return {"agent": "Engineering", "analysis": response.choices[0].message.content}
async def reporting_agent(analyses: list):    """Consolidates analyses from other agents into a single report."""    system_prompt = 'You are a reporting specialist. Consolidate the following analyses into a single, structured business report.'    content = json.dumps(analyses)    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": content}]
    )    return response.choices[0].message.content

## 3. Tool Definitions

Define the tools that the Operations Manager can use to take action.

In [ ]:
@function_tooldef log_action(action: str, details: str):    """Logs a decision or action taken by the Ops Manager."""    log_entry = f"[LOG] Action: {action} | Details: {details}"    print(log_entry)    return {"status": "logged", "entry": log_entry}
@function_tooldef generate_report(filename: str, content: str):    """Generates a structured report and saves it to a file."""    with open(filename, 'w') as f:        f.write(content)    print(f"[REPORT] Generated report: {filename}")    return {"status": "success", "path": filename}
@function_tooldef notify_stakeholders(message: str, channel: str):    """Simulates sending a notification to stakeholders."""    notification = f"[NOTIFICATION] To: {channel} | Message: {message}"    print(notification)    return {"status": "sent", "notification": notification}

## 4. Ops Manager Logic (Planner Agent)

This is the main orchestration logic. The Ops Manager receives a task, delegates work to other agents, and uses tools to take action.

In [ ]:
async def run_ops_manager(task: str):    # 1. Planning Phase    log_action(action="Task Received", details=task)    
    # 2. Parallel Agent Execution    log_action(action="Delegating to Specialized Agents", details="Finance, HR, Engineering")    analyses = await asyncio.gather(        finance_agent(task),        hr_agent(task),        engineering_agent(task)    )    print("--- Agent Analyses Complete ---")    for analysis in analyses:        print(f"{analysis["agent"]}: {analysis["analysis"][:100]}...")    
    # 3. Handoff and Consolidation    log_action(action="Handoff to Reporting Agent", details="Consolidating analyses.")    final_report_content = await reporting_agent(analyses)    print("--- Final Report Content ---")    print(final_report_content)    
    # 4. Final Action    log_action(action="Generating Final Report", details="report.txt")    generate_report(filename="report.txt", content=final_report_content)    log_action(action="Notifying Stakeholders", details="management")    notify_stakeholders(message=f"The report for task '{task}' is complete.", channel="management")    
    return final_report_content

## 5. Execute the System

Run the entire agentic workflow with a sample business task.

In [ ]:
business_task = "Launch a new feature: AI-powered code review for our enterprise customers."
# Run the async functionfinal_report = await run_ops_manager(business_task)

## 6. Trace and Observability Notes

The output of the `log_action` tool provides a clear, step-by-step trace of the Operations Manager's decisions. This is crucial for debugging, auditing, and understanding the agent's behavior in a real-world application.